# NutriPredict Congo: Analyse Exploratoire des Données (EDA)
**Auteur :** MPOY Schekina Lutte-de-vie  
**Date :** 08 juillet 2026  
**Notebook :** 03 / 05 — Analyse exploratoire  

---

## Objectif de ce notebook

Comprendre la distribution du stunting (retard de croissance)
selon les facteurs socio-démographiques disponibles dans le
dataset congo_clean.csv produit en Phase 2.

Questions auxquelles ce notebook répond :
1. Quelle est la distribution globale du stunting ?
2. Le stunting varie-t-il selon le milieu (urbain/rural) ?
3. Quel est le lien avec le niveau de richesse du ménage ?
4. L'accès à l'eau et à l'électricité joue-t-il un rôle ?
5. Y a-t-il des différences selon le sexe et l'âge de l'enfant ?
6. Le rang de naissance influence-t-il le risque ?

## Source des données
* Dataset : data/processed/congo_clean.csv
* Produit par : notebook 02_nettoyage_fusion.ipynb
* Lignes : 4 475 enfants mesurés
* Colonnes : 16 variables

## 1. Imports et chargement des données

On charge directement le fichier congo_clean.csv produit
en Phase 2, pas les fichiers DHS bruts. C'est l'avantage
d'avoir exporté un dataset propre : les notebooks suivants
n'ont plus besoin de refaire tout le nettoyage.

In [26]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import os  
import warnings
warnings.filterwarnings("ignore")

In [27]:
# configuration du style glabal des graphiques
sns.set_theme(style="whitegrid",palette="muted")
plt.rcParams["figure.dpi"]=120
plt.rcParams["font.family"]="sans-serif"

In [28]:
# Chargement du dataset 
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(),".."))
CHEMIN_DATA = os.path.join(BASE_DIR,"data","processed","congo_clean.csv")

df=pd.read_csv(CHEMIN_DATA)

print(f"Dataset chargé : {df.shape[0]} lignes,  {df.shape[1]} colonnes ")
print("\n == Aperçu du dataset == ")
print(df.sample(5))
print("\n == types des colonnes ==")
print(df.info())

Dataset chargé : 4475 lignes,  16 colonnes 

 == Aperçu du dataset == 
      v001  v002  hw70  hw71  hw72  b4   b8  bord cle_menage  hv025  hv270  \
2456   192    30  0.04 -0.45 -0.71   1  2.0     7     192_30      2      2   
267     17    28 -2.01 -0.91  0.37   2  3.0     3      17_28      2      2   
1901   146    10 -3.40 -2.64 -0.55   2  0.0     3     146_10      2      1   
4288   364    24 -2.47 -1.16  0.40   2  3.0     2     364_24      1      2   
3776   295    18 -0.32  0.28  0.67   1  4.0     1     295_18      1      5   

      hv201  hv205  hv206  hv219  stunting  
2456     21     23      1      1         0  
267      43     23      1      1         1  
1901     42     23      0      1         1  
4288     41     23      0      1         1  
3776     12     12      1      1         0  

 == types des colonnes ==
<class 'pandas.DataFrame'>
RangeIndex: 4475 entries, 0 to 4474
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      ---------

## 2. Recodage des variables catégorielles

Les variables socio-démographiques sont stockées sous forme
de codes numériques (ex: hv025 = 1 ou 2). Pour que les
graphiques soient lisibles directement, on crée des versions
textuelles de ces variables.

On ne modifie pas les colonnes originales, on ajoute de
nouvelles colonnes avec le suffixe _label. Les colonnes
numériques originales restent disponibles pour le ML en Phase 4.

In [37]:
# Recodage des variables catégorielles en labels lisibles

#
df["hv025_label"] = df["hv025"].map({1:"Urbain",2:"Rural"})

df["hv270_label"] = df["hv270"].map({
    1:"Q1-très pauvre",
    2:"Q2-pauvre",
    3:"Q3-moyen",
    4:"Q4-riche",
    5:"Q5-très riche"
})

#Accès à l'electricité
df["hv206_label"] = df["hv206"].map({0:"Sans électricité",1:"Avec électricité",9:None})

# Sexe de l'enfant
df["b4_label"] = df["b4"].map({1:"Masculin",2:"Féminin"})

# Sexe du chef de famille
df["hv219_label"] = df["hv219"].map({1:"Homme",2:"Femme"})


# Vérification 1
print("=== Vérification des recodage ===")
print(f"hv025_labels :{df["hv025_label"].value_counts().to_dict()}")
print(f"hv270_label : {df['hv270_label'].value_counts().to_dict()}")
print(f"hv206_label : {df['hv206_label'].value_counts().to_dict()}")
print(f"b4_label    : {df['b4_label'].value_counts().to_dict()}")
print(f"hv219_label : {df['hv219_label'].value_counts().to_dict()}")

#Vérification 2
print("\n=== Vérification qu'aucun code n'est resté non mappé ===")
for col in ['hv025_label', 'hv270_label', 'hv206_label', 'b4_label', 'hv219_label']:
    nan_count = df[col].isna().sum()
    if nan_count > 0:
        print(f" {col} : {nan_count} valeurs non mappées,devenu NaN")
    else:
        print(f" {col} : aucune valeur non mappée")



=== Vérification des recodage ===
hv025_labels :{'Rural': 3335, 'Urbain': 1140}
hv270_label : {'Q1-très pauvre': 1997, 'Q2-pauvre': 1243, 'Q3-moyen': 500, 'Q4-riche': 423, 'Q5-très riche': 312}
hv206_label : {'Sans électricité': 3531, 'Avec électricité': 943}
b4_label    : {'Masculin': 2283, 'Féminin': 2192}
hv219_label : {'Homme': 3695, 'Femme': 780}

=== Vérification qu'aucun code n'est resté non mappé ===
 hv025_label : aucune valeur non mappée
 hv270_label : aucune valeur non mappée
 hv206_label : 1 valeurs non mappées,devenu NaN
 b4_label : aucune valeur non mappée
 hv219_label : aucune valeur non mappée
